In [ ]:
!pip install transformers scikit-learn pandas

# New Section

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

In [ ]:
# =====================
# LOAD DATA
# =====================
df = pd.read_csv('/content/drive/MyDrive/SEM4/NLP/Tickets_Training_Dataset2.csv', encoding='latin-1')

In [ ]:
df.head()

In [ ]:
import torch
import torch.nn as nn
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd


In [ ]:

# df = pd.read_csv("dataset.csv")

# Encode labels
cat_encoder = LabelEncoder()
prio_encoder = LabelEncoder()
complaint_encoder = LabelEncoder()

df["cat_label"] = cat_encoder.fit_transform(df["Category"])
df["prio_label"] = prio_encoder.fit_transform(df["Priority"])
df["complaint_label"] = complaint_encoder.fit_transform(df["Complaint_Group"])


### Handling Single-Instance Categories for Stratified Split


In [ ]:
# Identify categories with only one instance
category_counts = df['cat_label'].value_counts()
single_instance_categories = category_counts[category_counts < 2].index

# Filter out rows with single-instance categories
df_filtered = df[~df['cat_label'].isin(single_instance_categories)]

# Re-run train-test split with the filtered DataFrame
# Ensure all target labels are passed if they are to be split

train_texts, val_texts, train_cat, val_cat, train_prio, val_prio, train_complaint, val_complaint = train_test_split(
    df_filtered["ï»¿Tickets"],
    df_filtered["cat_label"],
    df_filtered["prio_label"],
    df_filtered["complaint_label"],
    test_size=0.2,
    stratify=df_filtered["cat_label"],
    random_state=42 # Added random_state for reproducibility
)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

# Define device before use
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Compute class weights for 'Category' ---
unique_train_cat_labels = np.unique(train_cat.values)
# Compute weights only for classes present in training data
temp_weights_cat = compute_class_weight(
    class_weight='balanced',
    classes=unique_train_cat_labels,
    y=train_cat.values
)
# Create a full weight tensor with size equal to total number of classes
class_weights_cat = torch.zeros(len(cat_encoder.classes_), dtype=torch.float)
for label, weight in zip(unique_train_cat_labels, temp_weights_cat):
    class_weights_cat[label] = weight
class_weights_cat = class_weights_cat.to(device)


# --- Compute class weights for 'Priority' ---
unique_train_prio_labels = np.unique(train_prio.values)
temp_weights_prio = compute_class_weight(
    class_weight='balanced',
    classes=unique_train_prio_labels,
    y=train_prio.values
)
class_weights_prio = torch.zeros(len(prio_encoder.classes_), dtype=torch.float)
for label, weight in zip(unique_train_prio_labels, temp_weights_prio):
    class_weights_prio[label] = weight
class_weights_prio = class_weights_prio.to(device)


# --- Compute class weights for 'Complaint_Group' ---
unique_train_complaint_labels = np.unique(train_complaint.values)
temp_weights_complaint = compute_class_weight(
    class_weight='balanced',
    classes=unique_train_complaint_labels,
    y=train_complaint.values
)
class_weights_complaint = torch.zeros(len(complaint_encoder.classes_), dtype=torch.float)
for label, weight in zip(unique_train_complaint_labels, temp_weights_complaint):
    class_weights_complaint[label] = weight
class_weights_complaint = class_weights_complaint.to(device)

In [ ]:

# TOKENIZER

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

train_enc = tokenize(train_texts)
val_enc = tokenize(val_texts)


In [ ]:
# MODEL
# Forward pass
# Outputs logits

class MultiTaskModel(nn.Module):
    def __init__(self, num_cat, num_prio, num_complaint):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")

        # Freeze most layers
        for param in self.bert.parameters():
            param.requires_grad = False

        # Unfreeze last transformer layer
        for param in self.bert.transformer.layer[-1].parameters():
            param.requires_grad = True

        hidden_size = self.bert.config.hidden_size

        self.dropout = nn.Dropout(0.3)
        self.cat_head = nn.Linear(hidden_size, num_cat)
        self.prio_head = nn.Linear(hidden_size, num_prio)
        self.complaint_head = nn.Linear(hidden_size, num_complaint) # New complaint head

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0]  # CLS token

        x = self.dropout(cls_output)
        cat_logits = self.cat_head(x)
        prio_logits = self.prio_head(x)
        complaint_logits = self.complaint_head(x) # New complaint logits

        return cat_logits, prio_logits, complaint_logits # Return all logits

In [ ]:
model = MultiTaskModel(
    num_cat=len(cat_encoder.classes_),
    num_prio=len(prio_encoder.classes_),
    num_complaint=len(complaint_encoder.classes_)
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-5)

# Define separate loss functions for each task, incorporating class weights
loss_fn_cat = nn.CrossEntropyLoss(weight=class_weights_cat)
loss_fn_prio = nn.CrossEntropyLoss(weight=class_weights_prio)
loss_fn_complaint = nn.CrossEntropyLoss(weight=class_weights_complaint) # New loss function for complaint

In [ ]:
# Convert labels to tensors

# Function to safely get a numpy array from a pandas Series or a torch Tensor
def get_numpy_array_from_labels(label_data):
    if isinstance(label_data, pd.Series):
        return label_data.values
    elif isinstance(label_data, torch.Tensor):
        # If it's already a tensor, get its numpy representation
        return label_data.cpu().numpy()
    else:
        # Should not happen given previous steps, but good for robustness
        raise TypeError(f"Unsupported label data type: {type(label_data)}")

train_cat = torch.tensor(get_numpy_array_from_labels(train_cat)).long()
train_prio = torch.tensor(get_numpy_array_from_labels(train_prio)).long()
train_complaint = torch.tensor(get_numpy_array_from_labels(train_complaint)).long() # Convert train_complaint
val_cat = torch.tensor(get_numpy_array_from_labels(val_cat)).long()
val_prio = torch.tensor(get_numpy_array_from_labels(val_prio)).long()
val_complaint = torch.tensor(get_numpy_array_from_labels(val_complaint)).long() # Convert val_complaint

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# TRAINING

# Create TensorDatasets
train_dataset = TensorDataset(
    train_enc['input_ids'],
    train_enc['attention_mask'],
    train_cat,
    train_prio,
    train_complaint
)

val_dataset = TensorDataset(
    val_enc['input_ids'],
    val_enc['attention_mask'],
    val_cat,
    val_prio,
    val_complaint
)

# Define DataLoader
batch_size = 16 # Reduce batch size to prevent OOM errors
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

EPOCHS = 20

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0

    for batch in train_dataloader:
        optimizer.zero_grad()

        input_ids, attention_mask, cat_labels, prio_labels, complaint_labels = [
            t.to(device) for t in batch
        ]

        cat_logits, prio_logits, complaint_logits = model(input_ids, attention_mask)

        # Use the appropriate loss function for each task
        loss_cat = loss_fn_cat(cat_logits, cat_labels)
        loss_prio = loss_fn_prio(prio_logits, prio_labels)
        loss_complaint = loss_fn_complaint(complaint_logits, complaint_labels)

        loss = loss_cat + loss_prio + loss_complaint # Sum all losses
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_dataloader)

    # Validation phase
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_dataloader:
            val_input_ids, val_attention_mask, val_cat_labels, val_prio_labels, val_complaint_labels = [
                t.to(device) for t in batch
            ]

            val_cat_logits, val_prio_logits, val_complaint_logits = model(val_input_ids, val_attention_mask)

            val_loss_cat = loss_fn_cat(val_cat_logits, val_cat_labels)
            val_loss_prio = loss_fn_prio(val_prio_logits, val_prio_labels)
            val_loss_complaint = loss_fn_complaint(val_complaint_logits, val_complaint_labels)

            val_loss = val_loss_cat + val_loss_prio + val_loss_complaint
            total_val_loss += val_loss.item()

    avg_val_loss = total_val_loss / len(val_dataloader)

    print(f"Epoch {epoch+1} Train Loss: {avg_train_loss:.4f}, Validation Loss: {avg_val_loss:.4f}")

In [ ]:
# =====================
# EVALUATION
# =====================
model.eval()


In [ ]:
# model.eval()

with torch.no_grad():
    input_ids = val_enc["input_ids"].to(device)
    attention_mask = val_enc["attention_mask"].to(device)

    cat_logits, prio_logits, complaint_logits = model(input_ids, attention_mask) # Get complaint_logits

    cat_preds = torch.argmax(cat_logits, dim=1).cpu()
    prio_preds = torch.argmax(prio_logits, dim=1).cpu()
    complaint_preds = torch.argmax(complaint_logits, dim=1).cpu() # Get complaint predictions

### Classification Report for Complaint Group

In [ ]:
print("\nComplaint Group Report:")
print(classification_report(
    val_complaint,
    complaint_preds,
    labels=list(range(len(complaint_encoder.classes_))),
    target_names=complaint_encoder.classes_,
    zero_division=0
))

In [ ]:
from sklearn.metrics import classification_report

# Define labels separately
cat_labels = list(range(len(cat_encoder.classes_)))
prio_labels = list(range(len(prio_encoder.classes_)))
category_counts = df['cat_label'].value_counts()
single_instance_categories = category_counts[category_counts < 2].index

# Filter out rows with single-instance categories
df_filtered = df[~df['cat_label'].isin(single_instance_categories)]


In [ ]:
print("\nCategory Report:")
print(classification_report(
    val_cat,
    cat_preds,
    labels=cat_labels,
    target_names=cat_encoder.classes_,
    zero_division=0
))


In [ ]:
print("\nPriority Report:")
print(classification_report(
    val_prio,
    prio_preds,
    labels=prio_labels,
    target_names=prio_encoder.classes_,
    zero_division=0
))


### Data Distribution: Category, Priority, and Complaint Group Counts

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot Category Counts
plt.figure(figsize=(10, 6))
ax = sns.countplot(data=df, x='Category', hue='Category', palette='viridis', legend=False)
plt.title('Distribution of Categories')
plt.xlabel('Category')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')

# Add labels to the bars
for p in ax.patches:
    ax.annotate(f'{p.get_height()}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', fontsize=9, color='black', xytext=(0, 5),
                textcoords='offset points')

plt.tight_layout()
plt.show()

In [ ]:
# Plot Priority Counts
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df, x='Priority', hue='Priority', palette='magma', legend=False)
plt.title('Distribution of Priorities')
plt.xlabel('Priority')
plt.ylabel('Count')

# Add labels to the bars
for p in ax.patches:
    ax.annotate(f'{p.get_height()}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', fontsize=9, color='black', xytext=(0, 5),
                textcoords='offset points')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot Complaint_Group Counts
plt.figure(figsize=(12, 6))
ax = sns.countplot(data=df, x='Complaint_Group', hue='Complaint_Group', palette='plasma', legend=False)
plt.title('Distribution of Complaint Groups')
plt.xlabel('Complaint Group')
plt.ylabel('Count')
plt.xticks(rotation=60, ha='right')

# Add labels to the bars
for p in ax.patches:
    ax.annotate(f'{p.get_height()}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', fontsize=9, color='black', xytext=(0, 5),
                textcoords='offset points')

plt.tight_layout()
plt.show()

In [ ]:
print("Category classes:", cat_encoder.classes_)
print("Priority classes:", prio_encoder.classes_)
print("Complaint_Group classes:", complaint_encoder.classes_)

In [ ]:
### TEST DataSet

In [ ]:
# =====================
# LOAD TEST UNSEEN DATA
# =====================
df_test = pd.read_csv('/content/drive/MyDrive/SEM4/NLP/Tickets_Test_Dataset2.csv', encoding='latin-1')
display(df_test.head())

In [ ]:
def clean_col_names(df):
    # Remove BOM if present and strip whitespace
    df.columns = [col.replace('ï»¿', '').strip() for col in df.columns]
    return df

df_test = clean_col_names(df_test)

# Ensure 'Tickets' column exists after cleaning
if 'Tickets' not in df_test.columns:
    # Fallback if the cleaning didn't result in 'Tickets'
    # This part can be adjusted based on common alternative names like 'Query'
    if 'Query' in df_test.columns:
        df_test.rename(columns={'Query': 'Tickets'}, inplace=True)
    else:
        raise ValueError("Neither 'Tickets' nor 'Query' column found in test data after cleaning.")

# Filter out unseen labels in test set if necessary
# This logic ensures that `test_cat`, `test_prio`, `test_complaint` have consistent lengths

# Collect unique labels from training data encoders
valid_cat_labels = set(range(len(cat_encoder.classes_)))
valid_prio_labels = set(range(len(prio_encoder.classes_)))
valid_complaint_labels = set(range(len(complaint_encoder.classes_)))

initial_test_rows = len(df_test)

# Process Category labels
df_test['cat_label'] = df_test['Category'].apply(lambda x: cat_encoder.transform([x])[0] if x in cat_encoder.classes_ else -1)
df_test = df_test[df_test['cat_label'] != -1]

# Process Priority labels
df_test['prio_label'] = df_test['Priority'].apply(lambda x: prio_encoder.transform([x])[0] if x in prio_encoder.classes_ else -1)
df_test = df_test[df_test['prio_label'] != -1]

# Process Complaint_Group labels (check if column exists and then process)
can_evaluate_complaint = False
complaint_col_name = None
if 'Complaint_Group' in df_test.columns:
    complaint_col_name = 'Complaint_Group'
    can_evaluate_complaint = True
elif 'Complaint' in df_test.columns:
    complaint_col_name = 'Complaint'
    can_evaluate_complaint = True

if can_evaluate_complaint:
    df_test['complaint_label'] = df_test[complaint_col_name].apply(lambda x: complaint_encoder.transform([x])[0] if x in complaint_encoder.classes_ else -1)
    df_test = df_test[df_test['complaint_label'] != -1]
    test_complaint = torch.tensor(df_test['complaint_label'].values).long()
else:
    print(f"Warning: '{complaint_col_name}' column not found in df_test. Complaint Group evaluation will be skipped.")
    test_complaint = torch.tensor([]) # Assign an empty tensor if not found


if len(df_test) < initial_test_rows:
    print(f"Filtered out {initial_test_rows - len(df_test)} rows from test data due to unseen labels.")

# Convert to tensors
test_cat = torch.tensor(df_test['cat_label'].values).long()
test_prio = torch.tensor(df_test['prio_label'].values).long()

print("df_test columns after cleaning and processing:", df_test.columns.tolist())
print("Length of test_cat (after filtering):", len(test_cat))
print("Length of test_prio (after filtering):", len(test_prio))
if can_evaluate_complaint:
    print("Length of test_complaint (after filtering):", len(test_complaint))


In [ ]:
# Tokenize the 'Tickets' column of the test dataset after cleaning in the previous cell.
test_enc = tokenize(df_test["Tickets"])

In [ ]:
# =====================
# EVALUATION ON TEST DATA
# =====================

# Dynamically create TensorDataset arguments based on available labels
dataset_args = [
    test_enc['input_ids'],
    test_enc['attention_mask'],
    test_cat,
    test_prio
]
can_evaluate_complaint_in_dataloader = (test_complaint is not None and len(test_complaint) > 0)
if can_evaluate_complaint_in_dataloader:
    dataset_args.append(test_complaint)

# Create TensorDataset for test data
test_dataset = TensorDataset(*dataset_args)

# Define DataLoader for test data
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

model.eval()

all_cat_preds = []
all_prio_preds = []
all_complaint_preds = []

with torch.no_grad():
    for batch in test_dataloader:
        input_ids, attention_mask = batch[0], batch[1]
        cat_labels = batch[2] # Always present
        prio_labels = batch[3] # Always present

        if can_evaluate_complaint_in_dataloader:
            complaint_labels = batch[4]
            input_ids, attention_mask, cat_labels, prio_labels, complaint_labels = [
                t.to(device) for t in (input_ids, attention_mask, cat_labels, prio_labels, complaint_labels)
            ]
            cat_logits, prio_logits, complaint_logits = model(input_ids, attention_mask)
            all_complaint_preds.extend(torch.argmax(complaint_logits, dim=1).cpu().numpy())
        else:
            input_ids, attention_mask, cat_labels, prio_labels = [
                t.to(device) for t in (input_ids, attention_mask, cat_labels, prio_labels)
            ]
            cat_logits, prio_logits, _ = model(input_ids, attention_mask) # Model still returns 3 outputs, ignore the third

        all_cat_preds.extend(torch.argmax(cat_logits, dim=1).cpu().numpy())
        all_prio_preds.extend(torch.argmax(prio_logits, dim=1).cpu().numpy())


cat_test_preds = np.array(all_cat_preds)
prio_test_preds = np.array(all_prio_preds)

# Only create complaint_test_preds if complaint labels were processed
if can_evaluate_complaint_in_dataloader:
    complaint_test_preds = np.array(all_complaint_preds)
else:
    complaint_test_preds = None # Indicate that this was skipped
    print("'complaint_test_preds' is not available as 'Complaint_Group' evaluation was skipped.")

In [ ]:
from sklearn.metrics import classification_report

# Define labels separately (already defined previously, but good to ensure)
cat_labels = list(range(len(cat_encoder.classes_)))
prio_labels = list(range(len(prio_encoder.classes_)))

print("\nCategory Report (Test Data):")
print(classification_report(
    test_cat,
    cat_test_preds,
    labels=cat_labels,
    target_names=cat_encoder.classes_,
    zero_division=0
))

In [ ]:
print("\nPriority Report (Test Data):")
print(classification_report(
    test_prio,
    prio_test_preds,
    labels=prio_labels,
    target_names=prio_encoder.classes_,
    zero_division=0
))

In [ ]:
if complaint_test_preds is not None:
    print("\nComplaint Group Report (Test Data):")
    print(classification_report(
        test_complaint, # Use test_complaint directly as it's now a tensor
        complaint_test_preds,
        labels=list(range(len(complaint_encoder.classes_))),
        target_names=complaint_encoder.classes_,
        zero_division=0
    ))
else:
    print("\nComplaint Group Report (Test Data): Skipped due to missing 'Complaint_Group' column in test data.")

### Confusion Matrix for Category

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Generate confusion matrix for Category
cm_cat = confusion_matrix(test_cat, cat_test_preds, labels=cat_labels)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_cat,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=cat_encoder.classes_,
    yticklabels=cat_encoder.classes_
)
plt.title('Confusion Matrix for Category (Test Data)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

### Confusion Matrix for Priority

In [ ]:
# Generate confusion matrix for Priority
cm_prio = confusion_matrix(test_prio, prio_test_preds, labels=prio_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm_prio,
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=prio_encoder.classes_,
    yticklabels=prio_encoder.classes_
)
plt.title('Confusion Matrix for Priority (Test Data)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

### Confusion Matrix for Complaint Group

In [ ]:
if complaint_test_preds is not None and len(test_complaint) > 0:
    # Generate confusion matrix for Complaint Group
    cm_complaint = confusion_matrix(test_complaint, complaint_test_preds, labels=list(range(len(complaint_encoder.classes_))))

    plt.figure(figsize=(12, 10))
    sns.heatmap(
        cm_complaint,
        annot=True,
        fmt='d',
        cmap='Purples',
        xticklabels=complaint_encoder.classes_,
        yticklabels=complaint_encoder.classes_
    )
    plt.title('Confusion Matrix for Complaint Group (Test Data)')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.xticks(rotation=90, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("Complaint Group Confusion Matrix: Skipped due to missing 'Complaint_Group' column in test data or no samples to evaluate.")

### AUC ROC Scores

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize

print("\nAUC ROC Scores (Test Data):")

# Helper function to get full logits for test data
def get_full_logits(model, dataloader, device):
    all_cat_logits = []
    all_prio_logits = []
    all_complaint_logits = []

    # Determine if complaint_labels are in the batch (based on test_dataset construction)
    # This needs to be done carefully as test_dataset could have 3 or 4 elements per batch
    first_batch_len = len(next(iter(dataloader)))
    has_complaint_labels = (first_batch_len == 5) # input_ids, attn_mask, cat_labels, prio_labels, complaint_labels

    with torch.no_grad():
        for batch in dataloader:
            input_ids_batch, attention_mask_batch = batch[0].to(device), batch[1].to(device)

            cat_logits_batch, prio_logits_batch, complaint_logits_batch = model(input_ids_batch, attention_mask_batch)

            all_cat_logits.append(cat_logits_batch)
            all_prio_logits.append(prio_logits_batch)
            all_complaint_logits.append(complaint_logits_batch)

    full_cat_logits = torch.cat(all_cat_logits, dim=0)
    full_prio_logits = torch.cat(all_prio_logits, dim=0)
    full_complaint_logits = torch.cat(all_complaint_logits, dim=0)

    return full_cat_logits, full_prio_logits, full_complaint_logits

# Get full logits for the test data
full_cat_logits, full_prio_logits, full_complaint_logits = get_full_logits(model, test_dataloader, device)

# --- Category AUC ROC ---
# Convert true labels to one-hot encoded format
cat_true_one_hot = label_binarize(test_cat.cpu().numpy(), classes=list(range(len(cat_encoder.classes_))))
# Get predicted probabilities for Category
cat_probs = torch.softmax(full_cat_logits, dim=1).cpu().numpy()

# Calculate AUC ROC for each class in Category (one-vs-rest) and then average
# Handle cases where a class might have no true instances (support=0) in the test set
roc_auc_cat = roc_auc_score(cat_true_one_hot, cat_probs, average='weighted', multi_class='ovr')
print(f"Category AUC ROC (weighted): {roc_auc_cat:.4f}")

# --- Priority AUC ROC ---
prio_true_one_hot = label_binarize(test_prio.cpu().numpy(), classes=list(range(len(prio_encoder.classes_))))
prio_probs = torch.softmax(full_prio_logits, dim=1).cpu().numpy()
roc_auc_prio = roc_auc_score(prio_true_one_hot, prio_probs, average='weighted', multi_class='ovr')
print(f"Priority AUC ROC (weighted): {roc_auc_prio:.4f}")

# --- Complaint Group AUC ROC ---
# The `get_full_logits` function already handles generating full_complaint_logits.
# We still need to check if 'test_complaint' is available and not empty.
if complaint_test_preds is not None and len(test_complaint) > 0:
    complaint_true_one_hot = label_binarize(test_complaint.cpu().numpy(), classes=list(range(len(complaint_encoder.classes_))))
    complaint_probs = torch.softmax(full_complaint_logits, dim=1).cpu().numpy()

    # Handle potential discrepancy in length if some samples were filtered out during testing data prep for complaint_group
    if len(complaint_probs) == len(test_complaint):
        roc_auc_complaint = roc_auc_score(complaint_true_one_hot, complaint_probs, average='weighted', multi_class='ovr')
        print(f"Complaint Group AUC ROC (weighted): {roc_auc_complaint:.4f}")
    else:
        print("Could not compute Complaint Group AUC ROC due to mismatch in lengths of predictions and true labels.")
else:
    print("Complaint Group AUC ROC: Skipped due to missing 'Complaint_Group' column in test data or no samples to evaluate.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd # Ensure pandas is imported

# Prepare data for plotting
auc_roc_data = {
    'Task': ['Category', 'Priority', 'Complaint Group'],
    'AUC ROC Score': [roc_auc_cat, roc_auc_prio, roc_auc_complaint]
}
auc_roc_df = pd.DataFrame(auc_roc_data)

plt.figure(figsize=(10, 6))
sns.barplot(x='Task', y='AUC ROC Score', data=auc_roc_df, hue='Task', palette='viridis', legend=False)
plt.title('AUC ROC Scores for Each Task (Test Data)')
plt.xlabel('Classification Task')
plt.ylabel('AUC ROC Score')
plt.ylim(0, 1) # AUC ROC scores are between 0 and 1

# Add labels to the bars
for index, row in auc_roc_df.iterrows():
    plt.text(index, row['AUC ROC Score'] + 0.02, round(row['AUC ROC Score'], 4), color='black', ha="center")

plt.tight_layout()
plt.show()

### AUC ROC Curves

In [ ]:
from sklearn.metrics import roc_curve, RocCurveDisplay

# --- Category ROC Curves ---
plt.figure(figsize=(10, 8))
for i, class_name in enumerate(cat_encoder.classes_):
    if i in np.unique(test_cat.cpu().numpy()): # Only plot for classes present in test_cat
        # Binarize true labels for the current class (one-vs-rest)
        y_true_bin = (test_cat.cpu().numpy() == i).astype(int)
        # Extract probabilities for the current class
        y_score = cat_probs[:, i]

        # Handle cases where roc_curve might return NaN due to only one class being present
        if len(np.unique(y_true_bin)) > 1:
            fpr, tpr, _ = roc_curve(y_true_bin, y_score)
            plt.plot(fpr, tpr, label=f'ROC curve of {class_name} (area = {roc_auc_score(y_true_bin, y_score):.2f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves for Category (One-vs-Rest)')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()

# --- Priority ROC Curves ---
plt.figure(figsize=(10, 8))
for i, class_name in enumerate(prio_encoder.classes_):
    if i in np.unique(test_prio.cpu().numpy()): # Only plot for classes present in test_prio
        y_true_bin = (test_prio.cpu().numpy() == i).astype(int)
        y_score = prio_probs[:, i]

        if len(np.unique(y_true_bin)) > 1:
            fpr, tpr, _ = roc_curve(y_true_bin, y_score)
            plt.plot(fpr, tpr, label=f'ROC curve of {class_name} (area = {roc_auc_score(y_true_bin, y_score):.2f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves for Priority (One-vs-Rest)')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()

# --- Complaint Group ROC Curves ---
if complaint_test_preds is not None and len(test_complaint) > 0:
    plt.figure(figsize=(12, 10))
    for i, class_name in enumerate(complaint_encoder.classes_):
        if i in np.unique(test_complaint.cpu().numpy()): # Only plot for classes present in test_complaint
            y_true_bin = (test_complaint.cpu().numpy() == i).astype(int)
            y_score = complaint_probs[:, i]

            if len(np.unique(y_true_bin)) > 1:
                fpr, tpr, _ = roc_curve(y_true_bin, y_score)
                plt.plot(fpr, tpr, label=f'ROC curve of {class_name} (area = {roc_auc_score(y_true_bin, y_score):.2f})')

    plt.plot([0, 1], [0, 1], 'k--', label='Random classifier')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves for Complaint Group (One-vs-Rest)')
    plt.legend(loc='lower right', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print("Complaint Group ROC Curves: Skipped due to missing 'Complaint_Group' column in test data or no samples to evaluate.")

### Save Trained Model and Encoders

In [ ]:
import torch
import joblib

# Define paths for saving
# You might want to save these to your Google Drive for persistence
model_save_path = '/content/drive/MyDrive/multi_task_ticket_classifier.pth'
cat_encoder_save_path = '/content/drive/MyDrive/cat_encoder.joblib'
prio_encoder_save_path = '/content/drive/MyDrive/prio_encoder.joblib'
complaint_encoder_save_path = '/content/drive/MyDrive/complaint_encoder.joblib'

# Save model state dictionary
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

# Save LabelEncoders
joblib.dump(cat_encoder, cat_encoder_save_path)
print(f"Category Encoder saved to {cat_encoder_save_path}")
joblib.dump(prio_encoder, prio_encoder_save_path)
print(f"Priority Encoder saved to {prio_encoder_save_path}")
joblib.dump(complaint_encoder, complaint_encoder_save_path)
print(f"Complaint Group Encoder saved to {complaint_encoder_save_path}")

### Demo: Real-time Ticket Classification


### Load Saved Model and Encoders for Demo

(To be used , if want to skip training and directly use the saved model.)

In [ ]:
import torch
import joblib
from transformers import DistilBertTokenizer

# Ensure the MultiTaskModel class definition is executed (e.g., from cell SYwIIZWmX_jn)
# if you are starting a fresh session and skipping previous cells.

# Define paths for loading (must match save paths)
model_load_path = '/content/drive/MyDrive/multi_task_ticket_classifier.pth'
cat_encoder_load_path = '/content/drive/MyDrive/cat_encoder.joblib'
prio_encoder_load_path = '/content/drive/MyDrive/prio_encoder.joblib'
complaint_encoder_load_path = '/content/drive/MyDrive/complaint_encoder.joblib'

# Load LabelEncoders
cat_encoder = joblib.load(cat_encoder_load_path)
prio_encoder = joblib.load(prio_encoder_load_path)
complaint_encoder = joblib.load(complaint_encoder_load_path)
print("LabelEncoders loaded successfully.")

# Re-initialize the model with the correct number of classes
# These lengths are derived from the loaded encoders
loaded_model = MultiTaskModel(
    num_cat=len(cat_encoder.classes_),
    num_prio=len(prio_encoder.classes_),
    num_complaint=len(complaint_encoder.classes_)
)

# Load the saved state dictionary
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_model.load_state_dict(torch.load(model_load_path, map_location=device))
loaded_model.eval() # Set to evaluation mode

# Move model to appropriate device
loaded_model.to(device)
model = loaded_model # Assign to 'model' variable for consistency with demo code

# Re-initialize the tokenizer
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

print("Model and Tokenizer loaded successfully.")

In [ ]:
while True:
    user_input = input("Enter a ticket description (or 'quit' to exit): ")

    if user_input.lower() == 'quit':
        print("Exiting demo.")
        break

    if user_input.strip() == "":
        print("Please enter a non-empty ticket description.")
        continue

    predicted_cat, predicted_prio, predicted_complaint = predict_ticket_attributes(
        user_input,
        model,
        tokenizer,
        device,
        cat_encoder,
        prio_encoder,
        complaint_encoder
    )

    print(f"\n--- Prediction Results ---")
    print(f"Original Ticket: {user_input}")
    print(f"Predicted Category: {predicted_cat}")
    print(f"Predicted Priority: {predicted_prio}")
    print(f"Predicted Complaint Group: {predicted_complaint}")
    print(f"--------------------------\n")
